# Laboratorio 1 · Entorno y primeros casos comprobados

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/udd-ia-generativa-2026/curso-ia/blob/lab01/notebooks/lab01.ipynb)

**Formativo, sin nota.** Trabajo individual en Google Colab: instalas el código del curso, ejecutas un caso común en dos modelos y entregas tu evidencia en Canvas.

Al terminar tendrás: una llamada a un modelo escrita por ti, casos ejecutados en dos modelos, una revisión de cada respuesta y una tabla con tokens, latencia y costo.

Antes de empezar, abre **Secretos** (el ícono de llave del panel izquierdo) y agrega tu clave de Gemini como `GOOGLE_API_KEY` o `GEMINI_API_KEY` (el nombre que usa el botón de Colab) y, si el docente te los entregó, `CURSO_API_KEY` y `CURSO_BASE_URL` del servidor del curso, con acceso para este notebook. El servidor del curso es un respaldo que solo funciona durante las clases de los viernes; Gemini es la ruta principal. En vez de los secretos del servidor también puedes descomentar las dos líneas de la celda de configuración y pegar la clave del día. No escribas tu clave de Gemini en las celdas ni imprimas `os.environ`.

Al instalar, pip puede avisar un conflicto de `google-auth` con `google-colab`: es esperado y no afecta al laboratorio. Si el docente publica una versión nueva durante la clase, usa **Entorno de ejecución → Reiniciar sesión** y vuelve a ejecutar desde la primera celda.

In [1]:
import os
import sys

# En Colab se instala el código del curso; en un repositorio con entorno propio ya está instalado.
if "COLAB_RELEASE_TAG" in os.environ:
    %pip install -q "curso-ia @ git+https://github.com/udd-ia-generativa-2026/curso-ia@lab01"
    # Reinstala solo el paquete del curso, por si quedó una copia anterior con la misma versión.
    %pip install -q --force-reinstall --no-deps "curso-ia @ git+https://github.com/udd-ia-generativa-2026/curso-ia@lab01"
    if "curso_ia" in sys.modules:
        print("El paquete ya estaba cargado: usa Entorno de ejecución → Reiniciar sesión y vuelve a ejecutar desde aquí.")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 945.6 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.5/142.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.1/151.1 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the 

In [2]:
import os
from pathlib import Path

import pandas as pd
from pydantic_ai import Agent

from curso_ia import (
    apply_review, blocked, create_model, lab_cases, load_config, parse_cases, retry_failed, review_template,
    run_case, save_run, status,
)

# Modelo de comparación: texto libre con la forma proveedor:modelo. Opciones recomendadas:
#   "google:gemma-4-26b-a4b-it"  Gemma 4, de pesos abiertos, gratis con tu clave de Gemini; puede tardar.
#   "curso:course-chat"          servidor del curso con Gemma 4 12B, solo durante las clases de los viernes;
#                                requiere los secretos CURSO_API_KEY y CURSO_BASE_URL.
COMPARISON_MODEL = "google:gemma-4-26b-a4b-it"
os.environ["COMPARISON_MODEL"] = COMPARISON_MODEL

# Servidor del curso sin Secretos: descomenta y pega la clave del día que entrega el docente.
# os.environ["CURSO_BASE_URL"] = "https://tofu.tail07b663.ts.net/v1"
# os.environ["CURSO_API_KEY"] = ""

config = load_config()
lab = Path("lab01")  # aquí quedan la evidencia y el archivo para Canvas
if config.mode == "simulated":
    print("MODO SIMULADO: respuestas simuladas; no acredita acceso a ningún modelo.")
for row in status(config):
    state = "listo" if row["ready"] else "falta: " + ", ".join(row["missing"])
    print(f"{row['role']:<11} {row['model']:<32} {state}")


generation  google:gemini-3.5-flash-lite     listo
comparison  google:gemma-4-26b-a4b-it        listo


La lista muestra qué modelo atiende cada rol (`proveedor:modelo`) y, si falta algo, el **nombre** de la variable; nunca su valor. `generation` usa Gemini y `comparison` usa el modelo que escribiste en `COMPARISON_MODEL`, en la celda anterior. Por defecto es Gemma 4, un modelo de pesos abiertos de Google que la API de Gemini sirve gratis, así que la comparación es entre un modelo comercial y uno abierto. Durante la clase puedes usar en su lugar el servidor del curso con `curso:course-chat`; fuera de clase ese servidor no responde. Si cambias `COMPARISON_MODEL`, vuelve a ejecutar esa celda. Con una clave propia también puedes cambiar `generation` con la variable `GENERATION_MODEL`.

## 1. Tu primera llamada

Un `Agent` de PydanticAI recibe un modelo y una instrucción; `run` envía la consulta y devuelve la respuesta (`output`) y el consumo (`usage`), con los tokens de entrada y de salida que determinan el costo. La celda usa el primer perfil que esté listo.

`run` es asíncrona: en un notebook se escribe `await agent.run(...)` directo en la celda, porque Jupyter y Colab ya tienen un bucle de eventos (asyncio) en marcha. El editor de Colab puede subrayar `await` con el aviso «allowed only within async function»: es un aviso del editor y la celda se ejecuta igual. `run_sync` no sirve aquí, justamente porque ese bucle ya está corriendo.

In [3]:
ready = [row["role"] for row in status(config) if row["ready"]]
role = ready[0] if ready else "generation"  # el primer perfil listo; si ninguno, muestra qué falta
print("Perfil:", role, config.models[role])
model = create_model(config.models[role], config)
agent = Agent(model, instructions="Responde en una sola frase.")
result = await agent.run("¿Qué es un token en un modelo de lenguaje?")
print(result.output)
print(result.usage)


Perfil: generation google:gemini-3.5-flash-lite
Un token es la unidad básica de texto, que puede ser una palabra, parte de ella o un signo de puntuación, en la que se divide la información para que el modelo pueda procesarla.
RunUsage(cost=Decimal('0.000106'), details={'text_prompt_tokens': 20}, input_text_tokens=20, input_tokens=20, output_tokens=40, requests=1)


## 2. Los casos

El caso común del curso viene en el paquete: la regla de tarifas por duelo del caso Air Canada en cuatro consultas, cada una con su respuesta esperada. Ejecútalo sin cambios.

Mínimo para hoy: el caso común en ambos perfiles; si el servidor del curso no está disponible, basta con Gemini y el bloqueo registrado. Extensión: en la celda siguiente, reemplaza el ejemplo de `MY_TASK` por dos o tres casos de una tarea que te interese (una instrucción y, por caso, una entrada y la respuesta que esperas), cambia `USE_MY_TASK` a `True` y vuelve a ejecutar desde aquí. Usa datos públicos o sintéticos.

In [4]:
MY_TASK = {
    "instruction": 'Clasifica el mensaje como "reclamo", "consulta" o "felicitación". Responde solo con la categoría.',
    "cases": [
        {"id": "ejemplo-1", "input": "¿A qué hora abre la sucursal de Providencia el sábado?", "expected": "consulta"},
        {"id": "ejemplo-2", "input": "Me cobraron dos veces la misma compra y nadie responde.", "expected": "reclamo"},
    ],
}
USE_MY_TASK = False  # extensión: True para ejecutar tus casos

cases = parse_cases(MY_TASK, "my-task") if USE_MY_TASK else lab_cases(1)
if cases.is_example:
    print("Aún usas los casos de ejemplo: reemplázalos por los de tu tarea.")
print("Instrucción:", cases.instruction)
pd.DataFrame([vars(case) for case in cases.cases])


Instrucción: Explica en qué información basas tu respuesta. Si no puedes confirmar una regla con la información disponible, indícalo.


,id,input,expected
0,sin-politica,Ya completé mi viaje. ¿Puedo solicitar ahora q...,No afirma una regla: indica que necesita la po...
1,con-politica,Documento para el ejercicio — síntesis didácti...,"No: según el documento, las solicitudes poster..."
2,antes-del-viaje,Documento para el ejercicio — síntesis didácti...,El documento solo excluye las solicitudes post...
3,dato-ausente,Documento para el ejercicio — síntesis didácti...,El documento no indica el porcentaje; hace fal...


## 3. Ejecutar en ambos perfiles

Cada caso se envía por separado, sin mensajes anteriores. Si un perfil no está configurado o falla, se registra el bloqueo y se sigue con el otro.

Si un caso queda en `error`, las columnas `error_type` y `http_status` dicen por qué; un 429 es el límite de solicitudes por minuto del nivel gratuito de Gemini: espera un minuto y vuelve a ejecutar. Un 503 indica alta demanda en Google: espera unos minutos y vuelve a ejecutar. Evita cambiar a un modelo Flash sin «Lite»: en el nivel gratuito solo admiten 20 solicitudes al día.

`SETTINGS` ajusta cómo responde el modelo, por ejemplo la temperatura o el máximo de tokens; ver [ajustes de modelo en PydanticAI](https://ai.pydantic.dev/agents/#model-run-settings) y la [lista de campos de `ModelSettings`](https://ai.pydantic.dev/api/settings/) (inglés).

Extensión: sube `REPETITIONS` a 2 o 3 para ver cuánto varían la respuesta, los tokens y la latencia.

In [5]:
from tqdm.auto import tqdm

REPETITIONS = 1
SETTINGS = {}  # por ejemplo {"temperature": 0.2}; vacío usa el valor de cada proveedor

rows = []
progress = tqdm(total=len(config.models) * len(cases.cases) * REPETITIONS, desc="Consultas")
for profile, ref in config.models.items():
    try:
        agent = Agent(create_model(ref, config), instructions=cases.instruction, model_settings=SETTINGS)
    except Exception as error:
        rows += [blocked(profile, ref, case, config, 1, error) for case in cases.cases]
        progress.update(len(cases.cases) * REPETITIONS)
        continue
    for case in cases.cases:
        for repetition in range(1, REPETITIONS + 1):
            progress.set_postfix_str(f"{profile} · {case.id}")
            rows.append(await run_case(agent, case, profile=profile, ref=ref, config=config, repetition=repetition))
            progress.update()
progress.close()

columns = [
    "profile", "case", "repetition", "status", "input_tokens", "output_tokens", "latency_ms",
    "estimated_cost_usd", "error_type", "http_status",
]
pd.DataFrame(rows).reindex(columns=columns)


Consultas:   0%|          | 0/8 [00:00<?, ?it/s]

,profile,case,repetition,status,input_tokens,output_tokens,latency_ms,estimated_cost_usd,error_type,http_status
0,generation,sin-politica,1,ok,47,108,998,0.000284,NaN,NaN
1,generation,con-politica,1,ok,102,120,897,0.000331,NaN,NaN
2,generation,antes-del-viaje,1,ok,102,188,1234,0.000501,NaN,NaN
3,generation,dato-ausente,1,ok,98,73,802,0.000212,NaN,NaN
4,comparison,sin-politica,1,ok,47,1337,29081,0.000000,NaN,NaN
5,comparison,con-politica,1,ok,102,579,13203,0.000000,NaN,NaN
6,comparison,antes-del-viaje,1,ok,102,923,20262,0.000000,NaN,NaN
7,comparison,dato-ausente,1,ok,98,475,10883,0.000000,NaN,NaN


### Reintentar las que fallaron

Si alguna consulta quedó en `error` (por ejemplo un 500, un 503 o un `TimeoutError`), la celda siguiente la vuelve a enviar hasta `MAX_RETRIES` veces, con una pausa de `WAIT_SECONDS` antes de cada intento para no chocar con el límite por minuto. Las consultas bloqueadas por falta de configuración no se reintentan. Si no hubo errores, la celda no envía nada.

La columna `attempts` muestra cuántos intentos usó cada consulta y queda en la evidencia: también es una medición de la confiabilidad de cada modelo. Puedes ejecutar la celda otra vez si aún quedan errores.

In [6]:
MAX_RETRIES = 2  # reintentos por consulta fallida
WAIT_SECONDS = 20  # pausa antes de cada reintento

rows = await retry_failed(rows, cases, config, settings=SETTINGS, max_retries=MAX_RETRIES, wait_s=WAIT_SECONDS)
pd.DataFrame(rows).reindex(columns=columns + ["attempts"])

,profile,case,repetition,status,input_tokens,output_tokens,latency_ms,estimated_cost_usd,error_type,http_status,attempts
0,generation,sin-politica,1,ok,47,108,998,0.000284,NaN,NaN,1
1,generation,con-politica,1,ok,102,120,897,0.000331,NaN,NaN,1
2,generation,antes-del-viaje,1,ok,102,188,1234,0.000501,NaN,NaN,1
3,generation,dato-ausente,1,ok,98,73,802,0.000212,NaN,NaN,1
4,comparison,sin-politica,1,ok,47,1337,29081,0.000000,NaN,NaN,1
5,comparison,con-politica,1,ok,102,579,13203,0.000000,NaN,NaN,1
6,comparison,antes-del-viaje,1,ok,102,923,20262,0.000000,NaN,NaN,1
7,comparison,dato-ausente,1,ok,98,475,10883,0.000000,NaN,NaN,1


### Leer las respuestas

La tabla anterior trae las mediciones. Aquí se ven las respuestas completas, agrupadas por caso: primero la respuesta esperada y luego lo que respondió cada perfil. Léelas antes de revisarlas en la sección 5.

In [7]:
from IPython.display import Markdown, display

expected = {case.id: case.expected for case in cases.cases}
display(Markdown(f"**Instrucción:** {cases.instruction}"))

for row in rows:
    if row["status"] != "ok" and row["case"] == cases.cases[0].id:
        reason = ", ".join(row.get("missing", [])) or row.get("error_type", row["status"])
        display(Markdown(f"> **{row['profile']}** ({row['model']}) no respondió: {reason}."))

for case in cases.cases:
    display(Markdown(f"---\n### {case.id}\n**Esperada:** {expected[case.id]}"))
    for row in rows:
        if row["case"] != case.id or row["status"] != "ok":
            continue
        cost = f" · {row['estimated_cost_usd']} USD" if row.get("estimated_cost_usd") is not None else ""
        display(Markdown(
            f"#### {row['profile']} · {row['model']} (repetición {row['repetition']})\n\n{row['response']}\n\n"
            f"<sub>{row['input_tokens']} tokens de entrada · {row['output_tokens']} de salida · "
            f"{row['latency_ms']} ms{cost}</sub>"
        ))


**Instrucción:** Explica en qué información basas tu respuesta. Si no puedes confirmar una regla con la información disponible, indícalo.

---
### sin-politica
**Esperada:** No afirma una regla: indica que necesita la política de la aerolínea para confirmarlo.

#### generation · google:gemini-3.5-flash-lite (repetición 1)

Basándome en la información disponible en este momento, no puedo confirmar si puedes solicitar la tarifa por duelo después de haber completado tu viaje, ya que no se proporciona ninguna regla, política o término sobre tarifas por duelo o reembolsos posteriores al viaje. 

Para responder con certeza, necesitaría consultar las políticas específicas de la empresa de transporte o aerolínea con la que viajaste. Te sugiero que te pongas en contacto directamente con su servicio de atención al cliente para verificar si permiten solicitudes retroactivas de esta tarifa.

<sub>47 tokens de entrada · 108 de salida · 998 ms · 0.0002841 USD</sub>

#### comparison · google:gemma-4-26b-a4b-it (repetición 1)

No puedo darte una respuesta definitiva de "sí" o "no" porque **no has especificado la aerolínea o la empresa de transporte** con la que realizaste el viaje. Las políticas varían drásticamente entre compañías.

Sin embargo, puedo explicarte en qué me baso para darte una orientación general y qué es lo que no puedo confirmar:

### 1. Información en la que baso mi respuesta (Prácticas generales de la industria)
Basándome en las políticas estándar de la mayoría de las aerolíneas y empresas de transporte de pasajeros, la aplicación de una "tarifa por duelo" (bereavement fare) suele seguir estas reglas:

*   **Solicitud previa al viaje:** La gran mayoría de las compañías exigen que la tarifa especial se solicite y se apruebe **antes de la compra del boleto o antes de que el viaje ocurra**. El objetivo de estas tarifas es ayudar con los costos imprevistos de un traslado urgente, no reembolsar la diferencia después de que el servicio ya fue prestado.
*   **Verificación de documentación:** Estas tarifas suelen requerir la presentación de un certificado de defunción o un documento legal que acredite el fallecimiento y el parentesco, generalmente en el momento de la reserva o durante el proceso de validación previo al vuelo.

### 2. Lo que NO puedo confirmar
**No puedo confirmar si la empresa específica que utilizaste permite solicitar la tarifa de forma retroactiva.**

Existen excepciones muy raras en algunas compañías que permiten solicitar un reembolso de la diferencia si se presenta la documentación inmediatamente después del suceso, pero **no es la norma general**. Sin conocer el nombre de la empresa, es imposible determinar si su política permite excepciones para viajes ya completados.

### Recomendación de acción
Para obtener una respuesta válida para tu caso particular, te sugiero lo siguiente:

1.  **Busca los "Términos y Condiciones"** en el sitio web de la empresa bajo los conceptos de "Bereavement Fare" o "Tarifas por duelo".
2.  **Contacta directamente al Servicio al Cliente** de la empresa. No intentes hacerlo a través de un buscador o de una agencia de viajes de terceros (como Expedia o Despegar), ya que ellos no gestionan las tarifas especiales de las aerolíneas; debes hablar con la compañía que opera el vuelo/viaje.
3.  **Ten la documentación lista:** Si decides llamar, ten a mano el certificado de defunción y los datos de tu reserva, ya que será lo primero que te pedirán para evaluar cualquier posibilidad de ajuste de tarifa.

<sub>47 tokens de entrada · 1337 de salida · 29081 ms · 0.0 USD</sub>

---
### con-politica
**Esperada:** No: según el documento, las solicitudes posteriores al viaje quedan excluidas.

...

#### generation · google:gemini-3.5-flash-lite (repetición 1)

Basándome en la **información disponible** en el documento del ejercicio (la síntesis didáctica de la regla histórica del caso Air Canada de 2024), la regla establece que: *“La política de tarifas por duelo no se aplica a las solicitudes de consideración presentadas después de completar el viaje.”*

Dado que **todavía no viajas** y estás solicitando la consideración hoy (es decir, antes de completar el viaje), la regla restrictiva mencionada en el documento (que se aplica a solicitudes *posteriores* al viaje) no impediría, en principio, tu solicitud. 

Sin embargo, **no puedo confirmar** con la información disponible si tu solicitud será aprobada o si cumples con todos los demás requisitos necesarios para obtener dicha tarifa, ya que el texto solo menciona el momento límite para presentarla (antes de completar el viaje) y no los criterios completos de elegibilidad.

<sub>102 tokens de entrada · 188 de salida · 1234 ms · 0.0005006 USD</sub>

#### comparison · google:gemma-4-26b-a4b-it (repetición 1)

**No puedo confirmar si te aplicarán la tarifa por duelo.**

**Explicación de la base de mi respuesta:**

Mi respuesta se basa exclusivamente en la lógica del texto proporcionado, el cual presenta una **condición de exclusión**, pero no una **condición de aplicación**.

1.  **Lo que el texto confirma:** Establece claramente un límite temporal para la *negación* del beneficio: si la solicitud se hace **después** de viajar, la política **no** aplica.
2.  **Lo que el texto no confirma:** El documento no describe los requisitos para que la política **sí** se aplique, ni establece un procedimiento para solicitudes realizadas **antes** del viaje.

En resumen: Aunque tu situación (no haber viajado aún) no cumple con la condición para que te denieguen la tarifa (haber completado el viaje), el texto es insuficiente para asegurar que la solicitud será aprobada, ya que no especifica las reglas de validación para viajes pendientes.

<sub>102 tokens de entrada · 923 de salida · 20262 ms · 0.0 USD</sub>

---
### dato-ausente
**Esperada:** El documento no indica el porcentaje; hace falta otra fuente.

#### generation · google:gemini-3.5-flash-lite (repetición 1)

Basándome en la información provista en el documento, **no puedo confirmar** qué porcentaje de descuento corresponde a la tarifa por duelo, ya que el texto no menciona ningún porcentaje ni detalla los beneficios económicos de dicha política. 

El documento solo establece que la política de tarifas por duelo no se aplica a las solicitudes presentadas después de completar el viaje.

<sub>98 tokens de entrada · 73 de salida · 802 ms · 0.0002119 USD</sub>

#### comparison · google:gemma-4-26b-a4b-it (repetición 1)

El documento proporcionado **no contiene la información necesaria** para responder a tu pregunta.

**Explicación:**
La información en la que baso mi respuesta es el texto íntegro que has facilitado: *"La política de tarifas por duelo no se aplica a las solicitudes de consideración presentadas después de completar el viaje"*.

Este texto únicamente establece una **condición temporal** para solicitar la tarifa (la solicitud debe hacerse antes de finalizar el viaje), pero no menciona cifras, valores monetarios ni porcentajes de descuento. Por lo tanto, no es posible confirmar cuál es el porcentaje de descuento con la información disponible.

<sub>98 tokens de entrada · 475 de salida · 10883 ms · 0.0 USD</sub>

## 4. Guardar la ejecución

La ejecución completa queda en un JSON dentro de `lab01/evidence/`, que no depende de las salidas del notebook: `cases.json` para el caso común y `my-task.json` para tus casos. Si vuelves a ejecutar, se reemplaza la ejecución anterior del mismo conjunto, con su revisión.

In [8]:
evidence_path = save_run(rows, cases, config, lab / "evidence", lab=1, settings=SETTINGS)
print("Ejecución guardada en", evidence_path)


Ejecución guardada en lab01/evidence/cases.json


## 5. Revisar cada respuesta

Ejecuta la celda siguiente: imprime un diccionario `REVIEW` con una entrada por respuesta y, encima de cada una, la respuesta esperada. Cópialo en la celda de abajo y completa cada par: `True` si la respuesta cumple o `False` si no, y un motivo en una frase (qué coincide, qué falta o qué se agregó sin respaldo). Relee las respuestas en «Leer las respuestas». Una respuesta incorrecta es tan útil como una correcta si explicas por qué.

In [9]:
print(review_template(evidence_path))


# Cómo completar: cada entrada es una respuesta del modelo.
#   A la izquierda, qué respuesta es: (perfil, caso, repetición). No la cambies.
#   A la derecha, tu revisión: (cumple, motivo).
#     cumple: True si la respuesta cumple la esperada o False si no; sin comillas y con mayúscula inicial.
#     motivo: una frase entre comillas que explique por qué.
#   Ejemplo: ('generation', 'con-politica', 1): (True, 'Dice que no y cita la regla del documento'),
REVIEW = {
    # Esperada: No afirma una regla: indica que necesita la política de la aerolínea para confirmarlo.
    ('generation', 'sin-politica', 1): (None, ''),
    # Esperada: No: según el documento, las solicitudes posteriores al viaje quedan excluidas.
    ('generation', 'con-politica', 1): (None, ''),
    # Esperada: El documento solo excluye las solicitudes posteriores al viaje; no confirma que se aplique antes ni sus requisitos.
    ('generation', 'antes-del-viaje', 1): (None, ''),
    # Esperada: El documento no indica el po

In [10]:
# Pega aquí el diccionario REVIEW y complétalo; puedes corregirlo y volver a ejecutar.
REVIEW = {}

evidence = apply_review(evidence_path, REVIEW, config)
print("Revisión completa:", evidence["review_complete"])


Revisión completa: False


Con la revisión completa, la celda siguiente resume cada perfil.

In [11]:
numeric = ["input_tokens", "output_tokens", "latency_ms", "estimated_cost_usd"]
reviewed = pd.DataFrame(evidence["results"]).reindex(columns=["profile", "status", "passed", *numeric])
reviewed[numeric] = reviewed[numeric].apply(pd.to_numeric, errors="coerce")
reviewed.groupby("profile").agg(
    responses=("status", lambda s: (s == "ok").sum()),
    passed=("passed", lambda s: (s == True).sum()),  # noqa: E712
    input_tokens=("input_tokens", "mean"),
    output_tokens=("output_tokens", "mean"),
    latency_ms=("latency_ms", "median"),
    cost_usd=("estimated_cost_usd", lambda s: s.sum(min_count=1)),
)


,responses,passed,input_tokens,output_tokens,latency_ms,cost_usd
profile,,,,,,
comparison,4,0,87.25,828.50,16732.5,0.000000
generation,4,0,87.25,122.25,947.5,0.001327


Lee la tabla con cuidado: con pocos casos no se puede concluir qué modelo es mejor; es una primera medición. El costo es una estimación: la fuente de cada precio queda en el JSON, y en el nivel gratuito de Gemini no pagas el precio de lista que se muestra.

## 6. Entregar y explicar

La celda siguiente comprime `lab01` con la ejecución y la revisión, y lo descarga. Descarga también este notebook con sus salidas (**Archivo → Descargar → Descargar .ipynb**). Los archivos de Colab se borran al cerrar la sesión: descarga antes de salir.

En la tarea del laboratorio 1 en Canvas, sube ambos archivos y explica en pocas líneas un caso que falló o un costo que te sorprendió, y qué cambiarías. Si usaste un asistente, declara su uso según las reglas del curso. Si ya tienes GitHub, también puedes guardar estos archivos en un repositorio propio; no es obligatorio.

Las parejas se forman después de esta clase; tu tarea y tus casos te servirán para proponer el tema del proyecto.

In [12]:
import shutil

archive = shutil.make_archive("lab01-submission", "zip", lab.parent, lab.name)
if "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import files
    files.download(archive)
print("Archivo para Canvas:", archive)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archivo para Canvas: /content/lab01-submission.zip
